In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import torch.nn.functional as F
import torch
import torch.utils.data as data
import torchvision.transforms.v2 as tfs
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm

In [ ]:
import numpy as np
import os
import random
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

In [ ]:
import json

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

In [ ]:
os.chdir('/content/drive/MyDrive/')

In [ ]:
SEED = 42

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
device

In [ ]:
random.seed(SEED)

# NumPy random
np.random.seed(SEED)
# PyTorch random (CPU)
torch.manual_seed(SEED)
# PyTorch random (GPU)
if torch.cuda.is_available():
  torch.cuda.manual_seed(SEED)
  torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

## Загрузка данных и настройка датасета и даталоадера

In [ ]:
dataset = np.load("project_listen_dataset.npz")
traffic_data = dataset['array1']    # Спектограммы (n, 256, 173)
labels_category = dataset['array2'] # Метки категорий
labels_target = dataset['array3']   # Таргет метки

In [ ]:
def split_data(data, labels_category, labels_target, test_size=0.15, val_size=0.15):
    """
    Разделяет данные на train/val/test

    Parameters:
      data: массив со спектограммами
      labels_category: метки категорий
      labels_target: таргет метки
    """
    train_data, test_data, train_category, test_category, train_target, test_target = train_test_split(
        data, labels_category, labels_target, test_size=test_size, random_state=SEED
    ) # первый раз разбиваем данные на train и test

    train_data, val_data, train_category, val_category, train_target, val_target = train_test_split(
        train_data, train_category, train_target, test_size=val_size, random_state=SEED
    ) # разбиваем train из предыдующего шага на еще один train и еще на val data
    # В итоге получается train, val и test выборки
    train_dataset = (train_data, train_category, train_target)
    val_dataset = (val_data, val_category, val_target)
    test_dataset = (test_data, test_category, test_target)

    return train_dataset, val_dataset, test_dataset

In [ ]:
train_dataset, val_dataset, test_dataset = split_data(traffic_data, labels_category, labels_target)

Решейпим для сверточной сети

In [ ]:
# Для формата Tensorflow
# train_data, train_category, train_target = train_dataset
# train_data = train_data.reshape(-1, 256, 173, 1)
# val_data, val_category, val_target = val_dataset
# val_data = val_data.reshape(-1, 256, 173, 1)
# test_data, test_category, test_target = test_dataset
# test_data = test_data.reshape(-1, 256, 173, 1)

In [ ]:
# Распаковываем
train_data, train_category, train_target = train_dataset
val_data, val_category, val_target = val_dataset
test_data, test_category, test_target = test_dataset

# Преобразуем в формат PyTorch: (n, channels, height, width)
train_data = train_data.reshape(-1, 1, 256, 173)  # (n, 1, 256, 173)
val_data = val_data.reshape(-1, 1, 256, 173)
test_data = test_data.reshape(-1, 1, 256, 173)

In [ ]:
print("train shape: ", train_data.shape)
print("val shape: ", val_data.shape)
print("test shape: ", test_data.shape)

In [ ]:
class AudioDataset(data.Dataset):
    def __init__(self, audio_data, categories, targets):
        self.audio_data = audio_data
        self.categories = categories
        self.targets = targets

    def __len__(self):
        return len(self.audio_data)

    def __getitem__(self, idx):
        audio_sample = self.audio_data[idx]
        category = self.categories[idx]
        target = self.targets[idx]

        audio_sample = torch.from_numpy(audio_sample).float()
        category = torch.tensor(category, dtype=torch.float)
        target = torch.tensor(target, dtype=torch.float)

        return audio_sample, category, target

In [ ]:
train_audio_dataset = AudioDataset(train_data, train_category, train_target)
train = data.DataLoader(train_audio_dataset, batch_size=32, shuffle=True)

In [ ]:
val_audio_dataset = AudioDataset(val_data, val_category, val_target)
validation = data.DataLoader(val_audio_dataset, batch_size=32, shuffle=False)

In [ ]:
test_audio_dataset = AudioDataset(test_data, test_category, test_target)
test = data.DataLoader(test_audio_dataset, batch_size=32, shuffle=False)

## Обучение модели

In [ ]:
def init_weights(m):
    if isinstance(m, nn.Conv2d):
        nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
        if m.bias is not None:
            nn.init.zeros_(m.bias)
    elif isinstance(m, nn.Linear):
        nn.init.xavier_uniform_(m.weight)
        if m.bias is not None:
            nn.init.zeros_(m.bias)

In [ ]:
num_categories = len(labels_category[0])
num_targets = len(labels_target[0])

class AudioCNN(nn.Module):
    def __init__(self, input_channels=1, height=256, width=173):
        super().__init__()

        # Сверточная часть
        self.features = nn.Sequential(
            # 1 -> 32
            nn.Conv2d(input_channels, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),  # 256x173 -> 128x86

            # Блок 2: 32 -> 64
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),  # 128x86 -> 64x43

            # 64 -> 128
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),  # 64x43 -> 32x21

            # 128 -> 256
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),  # 32x21 -> 16x10

            # 256 -> 256
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.ReLU(),
        )

        self.flatten = nn.Flatten()
        self.dropout = nn.Dropout(0.5)

        # Голова 1: для категории - тип трафика на звуке
        self.head_category = nn.Sequential(
            nn.Linear(256 * 16 * 10, 128),
            nn.ReLU(),
            nn.Linear(128, num_categories)
        )

        # Голова 2: для таргета - характер движения трафика на звуке
        self.head_target = nn.Sequential(
            nn.Linear(256 * 16 * 10, 128),
            nn.ReLU(),
            nn.Linear(128, num_targets)
        )

    def forward(self, x):
        # Общая часть
        x = self.features(x)
        x = self.flatten(x)
        x = self.dropout(x)

        # Две головы
        category_output = self.head_category(x)
        target_output = self.head_target(x)

        return category_output, target_output

In [ ]:
model = AudioCNN(
    input_channels=1,
    height=256,
    width=173
).to(device)
model.apply(init_weights)

In [ ]:
optimizer = optim.Adam(params=model.parameters(), lr=0.001, weight_decay=0.001)

# Функции потерь для каждой головы
criterion_category = nn.BCEWithLogitsLoss()
criterion_target = nn.BCEWithLogitsLoss()

epochs = 50
model.train()

In [ ]:
loss_lst_val = []  # список значений потерь при валидации
loss_lst = []      # список значений потерь при обучении

for _e in range(epochs):
    model.train()
    loss_mean = 0
    lm_count = 0

    train_tqdm = tqdm(train, leave=False)
    for x_train, y_category, y_target in train_tqdm:

        # Переносим данные на выбранное устройство
        x_train = x_train.to(device)
        y_category = y_category.to(device)
        y_target = y_target.to(device)

        # Forward pass - получаем два выхода
        predict_category, predict_target = model(x_train)

        # Считаем две потери
        loss_category = criterion_category(predict_category, y_category)
        loss_target = criterion_target(predict_target, y_target)

        # Общая потеря
        loss = loss_category + loss_target

        optimizer.zero_grad()
        loss.backward()

        optimizer.step()

        lm_count += 1
        loss_mean = 1/lm_count * loss.item() + (1 - 1/lm_count) * loss_mean
        train_tqdm.set_description(f"Epoch [{_e+1}/{epochs}], loss_mean={loss_mean:.3f}")

    # валидация модели
    model.eval()
    Q_val = 0
    count_val = 0

    val_tqdm = tqdm(validation, leave=False)
    for x_val, y_category, y_target in val_tqdm:
        with torch.no_grad():

            x_val = x_val.to(device)
            y_category = y_category.to(device)
            y_target = y_target.to(device)

            predict_category, predict_target = model(x_val)

            loss_category = criterion_category(predict_category, y_category)
            loss_target = criterion_target(predict_target, y_target)

            loss = loss_category + loss_target

            Q_val += loss.item()
            count_val += 1

    Q_val /= count_val

    loss_lst.append(loss_mean)
    loss_lst_val.append(Q_val)

    print(f" | loss_mean={loss_mean:.3f}, Q_val={Q_val:.3f}")

In [ ]:
Q_cat = 0
Q_target = 0

model.eval()

test_tqdm = tqdm(test, leave=False)
for x_test, y_category, y_target in test_tqdm:
    with torch.no_grad():
        #p = model(x_test)
        x_test = x_test.to(device)
        y_category = y_category.to(device)
        y_target = y_target.to(device)

        predict_category, predict_target = model(x_test)

        idx_cat_pred = torch.argmax(predict_category, dim=1)
        idx_cat_true = torch.argmax(y_category, dim=1)

        idx_target_pred = torch.argmax(predict_target, dim=1)
        idx_target_true = torch.argmax(y_target, dim=1)

        Q_cat += torch.sum(idx_cat_pred == idx_cat_true).item()
        Q_target += torch.sum(idx_target_pred == idx_target_true).item()

acc_cat = Q_cat / test_data.shape[0]
acc_target = Q_target / test_data.shape[0]
print(acc_cat, acc_target)

In [ ]:
torch.save(model.state_dict(), 'model_weights.pth')

## Проверка качества модели на тестовой выборке

In [ ]:
model.load_state_dict(torch.load('model_weights.pth', map_location=torch.device('cpu'), weights_only=True))

In [ ]:
category_mapping = {
    0: "car",
    1: "emv",
    2: "motorcycle",
    3: "tram",
    4: "truck"
}

target_mapping = {
    0: "acceleration",
    1: "bell",
    2: "braking",
    3: "horn",
    4: "idling",
    5: "passing",
    6: "siren"
}

In [ ]:
def evaluate_model(model, test_loader, device):
    """
    Оценивает модель на тестовой выборке.

    Args:
        model: PyTorch модель
        test_loader: DataLoader с тестовыми данными
        device: устройство (cuda/cpu)

    Returns:
        dict: Словарь с предсказаниями и целевыми значениями
    """
    model.eval()

    all_category_preds = []
    all_target_preds = []
    all_category_true = []
    all_target_true = []

    with torch.no_grad():
        for x_test, y_category, y_target in tqdm(test_loader, desc="Testing"):
            x_test = x_test.to(device)
            y_category = y_category.to(device)
            y_target = y_target.to(device)

            # Forward pass
            predict_category, predict_target = model(x_test)

            # Для BCEWithLogitsLoss применяем sigmoid
            predict_category = torch.sigmoid(predict_category)
            predict_target = torch.sigmoid(predict_target)

            # Получаем классы (argmax)
            cat_preds = torch.argmax(predict_category, dim=1)
            target_preds = torch.argmax(predict_target, dim=1)

            # Истинные классы (из one-hot)
            cat_true = torch.argmax(y_category, dim=1)
            target_true = torch.argmax(y_target, dim=1)

            # Сохраняем
            all_category_preds.extend(cat_preds.cpu().numpy())
            all_target_preds.extend(target_preds.cpu().numpy())
            all_category_true.extend(cat_true.cpu().numpy())
            all_target_true.extend(target_true.cpu().numpy())

    return {
        'category_preds': np.array(all_category_preds),
        'target_preds': np.array(all_target_preds),
        'category_true': np.array(all_category_true),
        'target_true': np.array(all_target_true),
    }

In [ ]:
def calculate_combined_accuracy(category_preds, target_preds, category_true, target_true):
  combined_accuracy = {}

  unique_categories = np.unique(category_true)
  unique_targets = np.unique(target_true)

  for category in unique_categories:
    for target in unique_targets:
      indices = np.where((category_true == category) & (target_true == target))[0]
      if len(indices) > 0:
        correct_predictions = np.sum((category_preds[indices] == category) & (target_preds[indices] == target))
        accuracy = correct_predictions / len(indices) * 100
        category_name = category_mapping.get(category, f"Category_{category}")
        target_name = target_mapping.get(target, f"Target_{target}")
        combined_accuracy[f"{category_name} & {target_name}"] = accuracy

  return combined_accuracy

In [ ]:
def calculate_class_accuracy(category_preds, target_preds, category_true, target_true):
    unique_categories = np.unique(category_true)
    unique_targets = np.unique(target_true)

    category_accuracy = {}
    for category in unique_categories:
        indices = np.where(np.argmax(test_category, axis=1) == category)[0]
        correct_predictions = np.sum(category_preds[indices] == category)
        accuracy = correct_predictions / len(indices) * 100 if len(indices) > 0 else 0
        category_name = category_mapping.get(category, str(category))
        category_accuracy[category_name] = accuracy

    target_accuracy = {}
    for target in unique_targets:
        indices = np.where(np.argmax(test_target, axis=1) == target)[0]
        correct_predictions = np.sum(target_preds[indices] == target)
        accuracy = correct_predictions / len(indices) * 100 if len(indices) > 0 else 0
        target_name = target_mapping.get(target, str(target))
        target_accuracy[target_name] = accuracy

    return {
        "category_accuracy": category_accuracy,
        "target_accuracy": target_accuracy
    }

In [ ]:
results = evaluate_model(model, test, device)

In [ ]:
class_accuracy = calculate_class_accuracy(results['category_preds'],
                              results['target_preds'],
                              results['category_true'],
                              results['target_true'])

In [ ]:
class_accuracy

In [ ]:
comb_accuracy = calculate_combined_accuracy(results['category_preds'],
                              results['target_preds'],
                              results['category_true'],
                              results['target_true'])

In [ ]:
comb_accuracy

In [ ]:
true_labels_category = results['category_true']
true_labels_target = results['target_true']
predicted_labels_category = results['category_preds']
predicted_labels_target = results['target_preds']

print("Classification for Categories:")
print(classification_report(true_labels_category, predicted_labels_category, target_names=list(category_mapping.values())))
print("Classification for Targets:")
print(classification_report(true_labels_target, predicted_labels_target, target_names=list(target_mapping.values())))

In [ ]:
conf_matrix_target = confusion_matrix(true_labels_target, predicted_labels_target)
plt.figure(figsize=(7, 5))
sns.heatmap(conf_matrix_target, annot=True, fmt='d', cmap='Blues', xticklabels=list(target_mapping.values()), yticklabels=list(target_mapping.values()))
plt.title('Confusion Matrix for Targets')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.show()

In [ ]:
conf_matrix_category = confusion_matrix(true_labels_category, predicted_labels_category)
plt.figure(figsize=(8, 5))
sns.heatmap(conf_matrix_category, annot=True, fmt='d', cmap='Blues', xticklabels=list(category_mapping.values()), yticklabels=list(category_mapping.values()))
plt.title('Confusion Matrix for Categories')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.show()

In [ ]:
true_labels = [
    f"{category_mapping[cat]}-{target_mapping[tar]}"
    for cat, tar in zip(true_labels_category, true_labels_target)
]

pred_labels = [
    f"{category_mapping[cat]}-{target_mapping[tar]}"
    for cat, tar in zip(predicted_labels_category, predicted_labels_target)
]

# 1. Создаем матрицу
unique_labels = sorted(set(true_labels + pred_labels))
conf_matrix = confusion_matrix(true_labels, pred_labels, labels=unique_labels)

print(f"Shape of conf_matrix: {conf_matrix.shape}")  # (n, n) - должно быть квадратной

# 2. Находим строки и колонки с ненулевыми значениями
non_empty_rows = np.any(conf_matrix != 0, axis=1)
non_empty_cols = np.any(conf_matrix != 0, axis=0)

print(f"Non-empty rows: {non_empty_rows.sum()}")
print(f"Non-empty cols: {non_empty_cols.sum()}")


# Используем один фильтр (пересечение)
keep_rows = non_empty_rows & non_empty_cols  # Метка есть и в строках, и в столбцах
keep_cols = keep_rows  # Используем тот же фильтр

conf_matrix_filtered = conf_matrix[keep_rows][:, keep_cols]
filtered_labels = [label for i, label in enumerate(unique_labels) if keep_rows[i]]

print(f"Filtered shape: {conf_matrix_filtered.shape}")  # Теперь квадратная!
print(f"Filtered labels count: {len(filtered_labels)}")

# 4. Визуализация
plt.figure(figsize=(10, 8))
conf_matrix_df = pd.DataFrame(
    conf_matrix_filtered,
    index=filtered_labels,
    columns=filtered_labels
)

sns.heatmap(
    conf_matrix_df,
    annot=True,
    fmt='d',
    cmap='Blues',
    annot_kws={'size': 10}
)
plt.title('Combined Confusion Matrix', fontsize=14)
plt.xlabel('Predicted', fontsize=12)
plt.ylabel('True', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
pip install grad-cam

## GradCAM

In [ ]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
import librosa

In [ ]:
def signal_preprocessing(file_path):
  y, sr = librosa.load(file_path, sr=None)
  mel_spectrogram = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=256, fmax=16384)
  mel_spectrogram_db = librosa.power_to_db(mel_spectrogram, ref=np.max)

  signal_tensor = torch.from_numpy(mel_spectrogram_db).float()
  signal_tensor = signal_tensor.unsqueeze(0)
  return signal_tensor

In [ ]:
category_mapping = {
    0: "car",
    1: "emv",
    2: "motorcycle",
    3: "tram",
    4: "truck"
}

target_mapping = {
    0: "acceleration",
    1: "bell",
    2: "braking",
    3: "horn",
    4: "idling",
    5: "passing",
    6: "siren"
}

In [ ]:
model.eval()
vehicle_class_names = ['car', 'truck', 'motorcycle', 'tram', 'emv']
action_class_names = ['acceleration', 'bell', 'braking', 'horn', 'idling', 'passing', 'siren']

In [ ]:
model_for_cam = nn.Sequential(
    model.features,
    model.flatten,
    nn.Identity()
).to('cpu')

In [ ]:
file_path = "./dataset/extracted/dataset/car/acceleration/car_acceleration_1.wav"
input_tensor = signal_preprocessing(file_path).to('cpu')

In [ ]:
input_tensor.reshape(-1, 1, 256, 173).shape

In [ ]:
with torch.no_grad():
    out_vehicle, out_action = model(input_tensor.reshape(-1, 1, 256, 173))
    pred_vehicle = torch.argmax(out_vehicle, dim=1).item()
    pred_action = torch.argmax(out_action, dim=1).item()

In [ ]:
target_layers = [model_for_cam[0][12]]

In [ ]:
data_labels_list = [0]
data_list = [file_path]
target_classes = list(range(5))


In [ ]:
for name, module in model.named_modules():
    print(f"{name}: {type(module).__name__}")

In [ ]:
cam = GradCAM(model=model, target_layers=target_layers)
targets = [ClassifierOutputTarget(pred_vehicle)]


In [ ]:
grayscale_cam = cam(input_tensor=input_tensor.reshape(-1, 1, 256, 173), targets=targets)
heatmap = grayscale_cam[0]

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Исходная спектрограмма
axes[0].imshow(input_tensor.squeeze(), cmap='viridis', aspect='auto', origin='lower')
axes[0].set_title('Original Spectrogram')
axes[0].set_xlabel('Time')
axes[0].set_ylabel('Frequency')

# Тепловая карта
im = axes[1].imshow(heatmap, cmap='jet', aspect='auto', origin='lower')
axes[1].set_title(f'Grad-CAM: Pred=Car')
axes[1].set_xlabel('Time')
axes[1].set_ylabel('Frequency')
plt.colorbar(im, ax=axes[1])

# Наложение
axes[2].imshow(input_tensor.squeeze(), cmap='viridis', aspect='auto', origin='lower', alpha=0.6)
axes[2].imshow(heatmap, cmap='jet', aspect='auto', origin='lower', alpha=0.4)
axes[2].set_title('Overlay')
axes[2].set_xlabel('Time')
axes[2].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

## Экспорт модели через ONNX

In [ ]:
print(num_categories)
print(num_targets)

In [ ]:
dummy_input = torch.randn(1, 1, 256, 173)

In [ ]:
model = AudioCNN(
    input_channels=1,
    height=256,
    width=173
)

model.load_state_dict(torch.load('model_weights.pth', map_location='cpu'))

In [ ]:
with torch.no_grad():
  torch.onnx.export(
      model,                      # модель
      dummy_input,                # пример входа
      'cnn_model.onnx',               # имя файла
      input_names=['input'],      # имена входов
      output_names=['category_output', 'target_output'],  # имена выходов
      dynamic_axes={              # динамические размеры
          'input': {0: 'batch_size'},
          'category_output': {0: 'batch_size'},
          'target_output': {0: 'batch_size'}
      },
      opset_version=11            # версия ONNX
  )

In [ ]:
pip install onnx onnxscript